# Chapter 3 — Local Projections
## Jordà (2005) LP vs VAR: Impulse Responses to a Monetary Policy Shock

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

This notebook replicates the **Local Projections** example from Chapter 3 using real quarterly US data from FRED. The central comparison is between two methods for tracing the dynamic effects of a monetary policy shock:

| Method | Core idea | Key advantage | Key limitation |
|---|---|---|---|
| **VAR($p$) IRF** | Estimate the full system; invert the MA representation | Efficient if correctly specified | Misspecification propagates into every horizon |
| **Local Projections (LP)** | Run a direct regression of $y_{t+h}$ on the shock at each horizon $h$ | Robust to misspecification; easy to add nonlinearities | Less efficient; wider confidence bands |

**Data (three individual FRED files):**

| File | Series | Frequency | Transformation |
|---|---|---|---|
| `GDPC1.xlsx` | Real GDP | Quarterly | HP-filtered cycle → output gap |
| `GDPDEF.xlsx` | GDP Deflator | Quarterly | $100 \times \ln$ → year-on-year change → inflation |
| `FEDFUNDS.xlsx` | Fed Funds Rate | Monthly → quarterly average | Level (%) |

**Estimation sample:** 1960:Q1 – 2007:Q4 (pre-ZLB, following Jordà 2005 and CEE 1999)

**Design conventions:**
- 🎛️ cells = tunable parameters — change and re-run freely
- Every code block has a **header comment** explaining *why* this step exists
- **Inline comments** explain *what* each non-trivial line does
- **"What to look for"** notes follow every figure
- Figures saved as `.png` and `.pdf` for LaTeX/Overleaf

---

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

from statsmodels.tsa.api import VAR
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 10,
})
print('All packages loaded.')

---
# Step 1 — Data Loading and Preparation

## File structure

Each FRED Excel file has two sheets:
- `README`: metadata (series description, units, update date)
- `Quarterly` (GDPC1, GDPDEF) or `Monthly` (FEDFUNDS): two columns — `observation_date` and the series value

FEDFUNDS is monthly and must be averaged to quarterly before merging. We use `resample('QE').mean()` which takes the within-quarter arithmetic average of the three monthly observations.

**Date normalisation:** FRED stores dates as the first day of the period (e.g. `1960-01-01` for 1960:Q1). After resampling, pandas uses quarter-end dates (`1960-03-31`). We normalise all series to quarter-end before merging.

In [ ]:
# 🎛️ FILE PATHS — change if files are in a different directory
PATH_GDP  = 'GDPC1.xlsx'
PATH_DEFL = 'GDPDEF.xlsx'
PATH_FF   = 'FEDFUNDS.xlsx'

# ── Load quarterly GDP and deflator ──────────────────────────────────────────
def load_fred_quarterly(path, name):
    """Load a FRED quarterly Excel file; return a date-indexed DataFrame."""
    df = pd.read_excel(path, sheet_name='Quarterly', header=0)
    df.columns = ['date', name]
    df['date'] = pd.to_datetime(df['date'])
    # Normalise FRED quarter-start dates to quarter-end for consistent merging
    df['date'] = df['date'].dt.to_period('Q').dt.to_timestamp('Q')
    return df.set_index('date')

gdp  = load_fred_quarterly(PATH_GDP,  'gdp')
defl = load_fred_quarterly(PATH_DEFL, 'deflator')
print(f'GDPC1   : {len(gdp)} obs,  {gdp.index[0].date()} → {gdp.index[-1].date()}')
print(f'GDPDEF  : {len(defl)} obs,  {defl.index[0].date()} → {defl.index[-1].date()}')

# ── Load monthly Fed funds and convert to quarterly average ──────────────────
ff_raw = pd.read_excel(PATH_FF, sheet_name='Monthly', header=0)
ff_raw.columns = ['date', 'fedfunds']
ff_raw['date'] = pd.to_datetime(ff_raw['date'])
# resample('QE').mean() takes the average of all months within each quarter
ff_q = ff_raw.set_index('date').resample('QE').mean()
print(f'FEDFUNDS: {len(ff_q)} quarterly obs,  {ff_q.index[0].date()} → {ff_q.index[-1].date()}')

## Variable construction

The three variables in the LP/VAR system are:

**1. Output gap** — deviation of log GDP from its HP trend:
$$\text{gap}_t = 100 \times (\ln Y_t - \ln Y_t^{\text{trend}})$$
The HP filter (Hodrick & Prescott, 1997) with $\lambda = 1600$ (the standard quarterly value) decomposes $\ln Y_t$ into a slow-moving trend and a cyclical component. The output gap measures the percent deviation of actual GDP from potential.

**Why HP filter rather than growth rates?** LP models are commonly estimated on *levels* or *gaps* rather than first differences, because the impulse response is expressed as a cumulative effect on the level of the variable. Using log-differences would require cumulating the IRF to recover the level effect — equivalent but more cumbersome.

**2. Inflation** — year-on-year change in log deflator:
$$\pi_t = 100 \times (\ln P_t - \ln P_{t-4})$$
Using a 4-quarter (year-on-year) change removes quarterly noise and aligns with how central banks monitor inflation. This is the standard measure in the empirical macro literature (e.g. Christiano, Eichenbaum & Evans, 1999).

**3. Federal Funds Rate** — level in percent (unchanged from FRED).

**Estimation sample:** 1960:Q1 – 2007:Q4, following the convention of excluding the zero-lower-bound (ZLB) period 2009–2015. During ZLB, the Fed funds rate is mechanically constrained at zero and cannot serve as an unrestricted measure of monetary policy stance.

In [ ]:
# 🎛️ SAMPLE AND TRANSFORMATION PARAMETERS
HP_LAMBDA   = 1600          # standard quarterly value (Hodrick-Prescott, 1997)
INFL_LAGS   = 4             # year-on-year inflation: diff(4) = 4-quarter log change
SAMPLE_START = '1960-01-01' # quarter-start date of estimation sample
SAMPLE_END   = '2007-12-31' # exclude ZLB; try '2019-12-31' to include pre-COVID era

# ── Merge all three series on the common quarterly date index ─────────────────
# inner join keeps only quarters present in all three files
df = gdp.join(defl, how='inner').join(ff_q, how='inner').dropna()

# ── Output gap via HP filter ──────────────────────────────────────────────────
df['log_gdp']    = 100 * np.log(df['gdp'])          # log GDP in percent
# hpfilter returns (cycle, trend); cycle is the output gap
cycle, trend     = hpfilter(df['log_gdp'], lamb=HP_LAMBDA)
df['output_gap'] = cycle                             # percent deviation from trend

# ── Year-on-year inflation ────────────────────────────────────────────────────
df['log_deflator'] = 100 * np.log(df['deflator'])   # log deflator in percent
# diff(4) computes the 4-quarter change: log P_t - log P_{t-4}
df['inflation']    = df['log_deflator'].diff(INFL_LAGS)

# ── Federal funds rate: level (already in %) ─────────────────────────────────
df['fed_funds'] = df['fedfunds']

# ── Restrict to estimation sample ────────────────────────────────────────────
df_sample = df.loc[SAMPLE_START:SAMPLE_END,
                   ['output_gap', 'inflation', 'fed_funds']].dropna()

first_q = f"{df_sample.index[0].year}:Q{df_sample.index[0].quarter}"
last_q  = f"{df_sample.index[-1].year}:Q{df_sample.index[-1].quarter}"

print(f'Estimation sample: {first_q} – {last_q}  (T = {len(df_sample)})')
print()
print(df_sample.describe().round(3))

**What you should see:** 192 observations. Output gap is centred near zero (HP cycle property). Inflation has a mean around 3.5–4%, reflecting the Great Inflation years that dominate the early sample. Fed funds rate averages around 5.5% in this pre-ZLB sample.

In [ ]:
# ── Figure 0: Data overview for the LP system ─────────────────────────────────
# Always inspect the variables before estimating — HP filter can create endpoint
# distortions, and year-on-year inflation introduces serial correlation.
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

plot_specs = [
    ('output_gap', 'Output Gap (%, HP filter  λ=1600)', 'steelblue'),
    ('inflation',  'Inflation (y/y %,  GDP Deflator)',  'darkred'),
    ('fed_funds',  'Federal Funds Rate (%)',             'darkgreen'),
]
for ax, (col, title, color) in zip(axes, plot_specs):
    ax.plot(df_sample.index, df_sample[col], color=color, linewidth=0.95)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('%')

# NBER recession shading
recessions = [
    ('1960-04-01','1961-01-01'), ('1969-10-01','1970-10-01'),
    ('1973-10-01','1975-01-01'), ('1980-01-01','1980-07-01'),
    ('1981-07-01','1982-10-01'), ('1990-07-01','1991-01-01'),
    ('2001-01-01','2001-10-01'),
]
for ax in axes:
    for s, e in recessions:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.11, color='grey')

axes[-1].set_xlabel('Quarter')
fig.suptitle(f'Figure 0: LP System Variables  |  {first_q} – {last_q}\n'
             f'Grey = NBER recessions  |  Output gap: HP(1600)  |  '
             f'Inflation: year-on-year GDP deflator',
             fontsize=12, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig0_lp_data.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** The output gap turns sharply negative during every NBER recession (grey shading) — this is the HP filter working as intended. Inflation spikes during 1973–1980 (oil shocks) and then falls sharply after Volcker's disinflation (1981–1982). The Fed funds rate tracks the inflation cycle closely, consistent with a Taylor rule. Note that using year-on-year inflation introduces a moving-average structure (the four-quarter change shares three overlapping quarters with adjacent observations) — this is one reason we use Newey-West standard errors in the LP regressions.

---
# Step 2 — VAR Estimation

We estimate two VARs as benchmarks:
- **VAR(4)**: the workhorse specification for quarterly macro data (Christiano, Eichenbaum & Evans, 1999 use $p=4$ for their baseline).
- **VAR(1)**: a deliberately mis-specified model used to illustrate how lag truncation distorts IRFs at long horizons.

**Cholesky identification:** We impose a recursive structure with ordering $(\text{output gap}, \pi, i)$. This means the Fed funds rate can react contemporaneously to output and inflation, but output and inflation do not respond to the funds rate within the same quarter — a timing assumption broadly consistent with sticky prices and implementation lags.

In [ ]:
# 🎛️ VAR PARAMETERS
VAR_P_MAIN   = 4    # main specification (CEE 1999 benchmark)
VAR_P_MISSP  = 1    # deliberately misspecified — shorter lags
IRF_HORIZON  = 20   # quarters ahead (5 years)

# Variable ordering for Cholesky: output_gap first (slowest to react),
# inflation second, fed_funds last (fastest/most endogenous)
CHOL_ORDER = ['output_gap', 'inflation', 'fed_funds']

# Indices in CHOL_ORDER
IDX_Y  = 0    # output gap
IDX_PI = 1    # inflation
IDX_FF = 2    # fed funds (shock variable)

print(f'Cholesky ordering: {CHOL_ORDER}')
print(f'Shock: {CHOL_ORDER[IDX_FF]}  (index {IDX_FF})')
print(f'Responses: {CHOL_ORDER[IDX_Y]} (index {IDX_Y}),  {CHOL_ORDER[IDX_PI]} (index {IDX_PI})')

In [ ]:
# ── Fit VAR(4) and VAR(1) ─────────────────────────────────────────────────────
# Data ordered according to Cholesky identification scheme
var_model   = VAR(df_sample[CHOL_ORDER])

# Check information criteria first
lag_sel = var_model.select_order(maxlags=8)
print(f'IC-selected lag orders (for information only):')
print(f'  AIC  → p = {lag_sel.aic}')
print(f'  BIC  → p = {lag_sel.bic}')
print(f'  HQIC → p = {lag_sel.hqic}')
print(f'\nWe use p = {VAR_P_MAIN} (CEE 1999 convention) and p = {VAR_P_MISSP} (misspecification demo)')

# Fit both models
var4 = var_model.fit(VAR_P_MAIN)
var1 = var_model.fit(VAR_P_MISSP)

print(f'\nVAR({VAR_P_MAIN}):  AIC = {var4.aic:.3f},  BIC = {var4.bic:.3f}')
print(f'VAR({VAR_P_MISSP}):  AIC = {var1.aic:.3f},  BIC = {var1.bic:.3f}')

In [ ]:
# ── Cholesky-identified IRFs from VAR(4) and VAR(1) ──────────────────────────
# irf(H).orth_irfs contains the orthogonalised IRFs from Cholesky decomposition
# orth_irfs[h, j, i] = response of variable j at horizon h to structural shock i
irf_var4 = var4.irf(IRF_HORIZON)
irf_var1 = var1.irf(IRF_HORIZON)

# Extract point estimates and standard errors for output gap and inflation
# responses to the Fed funds shock (IDX_FF)
var4_y   = irf_var4.orth_irfs[:, IDX_Y,  IDX_FF]    # output gap response
var4_pi  = irf_var4.orth_irfs[:, IDX_PI, IDX_FF]    # inflation response

var1_y   = irf_var1.orth_irfs[:, IDX_Y,  IDX_FF]
var1_pi  = irf_var1.orth_irfs[:, IDX_PI, IDX_FF]

# Analytical standard errors for the VAR(4) bands (delta method approximation)
var4_se_y  = irf_var4.stderr()[:, IDX_Y,  IDX_FF]
var4_se_pi = irf_var4.stderr()[:, IDX_PI, IDX_FF]

# 95% confidence bands for VAR(4)
var4_lo_y  = var4_y  - 1.96 * var4_se_y
var4_hi_y  = var4_y  + 1.96 * var4_se_y
var4_lo_pi = var4_pi - 1.96 * var4_se_pi
var4_hi_pi = var4_pi + 1.96 * var4_se_pi

print(f'VAR({VAR_P_MAIN}) Cholesky IRFs extracted.')
print(f'  Output gap h=0..5: {var4_y[:6].round(4)}')
print(f'  Inflation  h=0..5: {var4_pi[:6].round(4)}')
print(f'\nNote: h=0 output gap response should be near 0 (contemporaneous restriction).')

**What to look for:** Under the Cholesky ordering with Fed funds last, a monetary policy shock does *not* affect output or inflation at horizon $h=0$ — the contemporaneous restrictions impose this. The response should become negative for the output gap at $h=4$–$8$ (contractionary effect of a rate rise) and — if identification is working correctly — negative for inflation at $h=6$–$12$ (price puzzle resolved). If inflation goes *positive* initially before turning negative, this is the **price puzzle** (Sims, 1992), often attributed to missing information variables.

---
# Step 3 — Local Projections

## The LP estimator

Jordà (2005) proposes estimating the impulse response at each horizon $h$ **directly** by running a separate OLS regression:

$$y_{t+h} = \alpha_h + \beta_h^0 \, \text{shock}_t + \sum_{\ell=1}^{L} \Gamma_{h,\ell} Z_{t-\ell} + \varepsilon_{t+h}^h$$

where:
- $y_{t+h}$ is the response variable $h$ quarters ahead (shifted left by $h$)
- $\text{shock}_t$ is the contemporaneous value of the shock variable
- $Z_{t-\ell}$ are $L$ lags of **all** variables in the system (controls)
- $\hat{\beta}_h^0$ is the LP-IRF at horizon $h$

The IRF is simply the sequence $\{\hat{\beta}_0^0, \hat{\beta}_1^0, \hat{\beta}_2^0, \ldots, \hat{\beta}_H^0\}$ — one regression per horizon.

**Why Newey-West standard errors?** The regression error $\varepsilon_{t+h}^h$ is a moving average of order $h$ by construction (it includes the forecast errors from $t+1$ through $t+h$). Ignoring this autocorrelation would understate standard errors. Newey-West (HAC) covariance estimation with lag truncation $\max(h+1, L)$ corrects for this.

In [ ]:
# 🎛️ LP PARAMETERS
LP_LAGS = 4     # number of control lags L; should match VAR lag order for comparability
                # try LP_LAGS = 2 or 8 to check robustness

SHOCK_VAR = 'fed_funds'    # the variable whose shock we trace
CONF_LEVEL = 1.96          # 1.96 for 95% CI; use 1.645 for 90% CI

print(f'LP specification:')
print(f'  Control lags L = {LP_LAGS}')
print(f'  Shock variable = {SHOCK_VAR}')
print(f'  Horizon H = {IRF_HORIZON} quarters')
print(f'  Standard errors: Newey-West HAC, truncation = max(h+1, {LP_LAGS})')

In [ ]:
# ── Local Projections estimator ───────────────────────────────────────────────
# One OLS regression per horizon h = 0, 1, ..., H
# The slope on the contemporaneous shock is the IRF at horizon h

def estimate_lp(data, shock_var, response_var, H, n_lags):
    """
    Jordà (2005) Local Projections with Newey-West standard errors.

    At each horizon h, estimate:
        y_{t+h} = α + β_h * shock_t + Σ_ℓ Γ_ℓ Z_{t-ℓ} + ε_{t+h}

    β_h is the LP-IRF at horizon h.

    Parameters
    ----------
    data        : DataFrame with all system variables
    shock_var   : str — name of the shock variable (contemporaneous regressor)
    response_var: str — name of the response variable (dependent variable)
    H           : int — maximum forecast horizon
    n_lags      : int — number of control lags

    Returns
    -------
    irfs, ses, ci_lower, ci_upper : arrays of length H+1
    """
    irfs = np.zeros(H + 1)    # point estimates β_h
    ses  = np.zeros(H + 1)    # HAC standard errors

    for h in range(H + 1):
        # Dependent variable: y_{t+h} — shift response h periods into the future
        # (shift(-h) moves future values into the current row)
        y = data[response_var].shift(-h)

        # Control regressors: lags 1..n_lags of ALL variables in the system
        # Including lags of all variables prevents the shock from picking up
        # predictable variation driven by past dynamics
        X_list = [data[v].shift(lag)
                  for v in data.columns
                  for lag in range(1, n_lags + 1)]

        # Build regressor matrix: [shock_t, all_controls]
        X_df = pd.concat([data[shock_var]] + X_list, axis=1)
        X_df = add_constant(X_df)    # add intercept α_h

        # Drop all rows with any NaN (from shifts and sample edges)
        combined = pd.concat([y, X_df], axis=1).dropna()
        y_clean  = combined.iloc[:, 0].values
        X_clean  = combined.iloc[:, 1:].values

        # OLS with Newey-West HAC covariance
        # nw_lags: truncation parameter must be at least h+1 to account for the
        # MA(h) autocorrelation in ε_{t+h}; also at least n_lags as a floor
        nw_lags = max(h + 1, n_lags)
        res = OLS(y_clean, X_clean).fit(
            cov_type='HAC', cov_kwds={'maxlags': nw_lags}
        )

        # The shock variable is the first column AFTER the constant
        # add_constant places the constant at column 0, so shock is at index 1
        irfs[h] = res.params.iloc[1]    # β_h
        ses[h]  = res.bse.iloc[1]       # HAC standard error of β_h

    ci_lower = irfs - CONF_LEVEL * ses    # lower confidence band
    ci_upper = irfs + CONF_LEVEL * ses    # upper confidence band
    return irfs, ses, ci_lower, ci_upper

print('LP function defined. Starting estimation...')
print('(Three separate regressions: output gap, inflation, own-response)')

In [ ]:
# ── Run LP for all three response variables ───────────────────────────────────
# Each call fits H+1 = 21 OLS regressions; takes a few seconds

print('Estimating LP — output gap response...')
lp_y,   se_y,   lo_y,   hi_y   = estimate_lp(
    df_sample, SHOCK_VAR, 'output_gap', IRF_HORIZON, LP_LAGS)

print('Estimating LP — inflation response...')
lp_pi,  se_pi,  lo_pi,  hi_pi  = estimate_lp(
    df_sample, SHOCK_VAR, 'inflation',  IRF_HORIZON, LP_LAGS)

print('Estimating LP — own-response (Fed funds persistence)...')
lp_ff,  se_ff,  lo_ff,  hi_ff  = estimate_lp(
    df_sample, SHOCK_VAR, 'fed_funds',  IRF_HORIZON, LP_LAGS)

print('Done!')
print(f'\nLP output gap  h=0..5: {lp_y[:6].round(4)}')
print(f'LP inflation   h=0..5: {lp_pi[:6].round(4)}')
print(f'LP fed funds   h=0..5: {lp_ff[:6].round(4)}')

**What to look for:** `lp_ff[0]` should equal exactly 1.0 — the own-response of the Fed funds rate at $h=0$ is the contemporaneous shock itself. `lp_y[0]` should be small and positive (the impact effect of a rate rise on output, before the contractionary effect materialises). The LP confidence bands at long horizons are typically wider than VAR bands, reflecting the lower efficiency of direct projection at long forecast horizons.

---
# Step 4 — LP vs VAR Comparison

## Interpreting the comparison

When LP and VAR(4) give similar shapes but different magnitudes, the difference is efficiency: VAR(4) restricts the IRF to evolve according to the companion matrix dynamics, while LP is unrestricted. When LP and VAR(4) give **systematically different shapes**, this is evidence of **model misspecification** in the VAR — the recursive structure imposed by lag truncation is distorting the IRF at longer horizons.

VAR(1) serves as the misspecification straw man: by truncating at 1 lag, all long-run dynamics are forced through a single AR matrix, which badly approximates the true propagation mechanism.

In [ ]:
# ── Figure 1: Publication-quality LP vs VAR comparison ───────────────────────
# Side-by-side panels: output gap (left) and inflation (right)
# Each panel shows VAR(4) with bands, LP with bands, VAR(1) point estimate

horizons = np.arange(IRF_HORIZON + 1)

# Colour scheme: blue = VAR(4), red = LP, green = VAR(1)
C_VAR4 = '#2166AC'
C_LP   = '#B2182B'
C_VAR1 = '#4DAF4A'
ALPHA  = 0.13    # transparency for confidence bands

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(
    f'Figure 1: Impulse Responses to a Monetary Policy Shock — LP vs VAR\n'
    f'US Quarterly Data, {first_q}–{last_q}  |  '
    f'Identification: Cholesky ordering (output gap, inflation, fed funds)',
    fontsize=12, fontweight='bold'
)

# ── Panel (a): Output Gap ─────────────────────────────────────────────────────
ax = axes[0]

# VAR(4): confidence band then point estimate
ax.fill_between(horizons, var4_lo_y, var4_hi_y, color=C_VAR4, alpha=ALPHA)
ax.plot(horizons, var4_y, color=C_VAR4, linewidth=2.5,
        label=f'VAR({VAR_P_MAIN}) — Cholesky')

# LP: confidence band then point estimate (dashed)
ax.fill_between(horizons, lo_y, hi_y, color=C_LP, alpha=ALPHA)
ax.plot(horizons, lp_y, color=C_LP, linewidth=2.5, linestyle='--',
        label=f'Local Projections  (L={LP_LAGS})')

# VAR(1): dotted line only (misspecified benchmark)
ax.plot(horizons, var1_y, color=C_VAR1, linewidth=2.0, linestyle=':',
        label=f'VAR({VAR_P_MISSP}) — misspecified')

ax.axhline(0, color='black', linewidth=1.0)
ax.set_xlabel('Quarters after shock')
ax.set_ylabel('Percent of potential output')
ax.set_title('(a) Response of Output Gap')
ax.set_xlim(0, IRF_HORIZON)
ax.set_xticks([0, 4, 8, 12, 16, 20])
ax.legend(loc='lower right')

# Symmetric y-axis for clean presentation
ym = max(abs(np.array([var4_lo_y, var4_hi_y, lo_y, hi_y, var1_y])).max()) * 1.2
ax.set_ylim(-ym, ym)

# ── Panel (b): Inflation ──────────────────────────────────────────────────────
ax = axes[1]

ax.fill_between(horizons, var4_lo_pi, var4_hi_pi, color=C_VAR4, alpha=ALPHA)
ax.plot(horizons, var4_pi, color=C_VAR4, linewidth=2.5,
        label=f'VAR({VAR_P_MAIN}) — Cholesky')

ax.fill_between(horizons, lo_pi, hi_pi, color=C_LP, alpha=ALPHA)
ax.plot(horizons, lp_pi, color=C_LP, linewidth=2.5, linestyle='--',
        label=f'Local Projections  (L={LP_LAGS})')

ax.plot(horizons, var1_pi, color=C_VAR1, linewidth=2.0, linestyle=':',
        label=f'VAR({VAR_P_MISSP}) — misspecified')

ax.axhline(0, color='black', linewidth=1.0)
ax.set_xlabel('Quarters after shock')
ax.set_ylabel('Percentage points (year-on-year)')
ax.set_title('(b) Response of Inflation')
ax.set_xlim(0, IRF_HORIZON)
ax.set_xticks([0, 4, 8, 12, 16, 20])
ax.legend(loc='upper right')

ym_pi = max(abs(np.array([var4_lo_pi, var4_hi_pi, lo_pi, hi_pi, var1_pi])).max()) * 1.2
ax.set_ylim(-ym_pi, ym_pi)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig1_lp_vs_var.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for in panel (a) — output gap:** A contractionary monetary policy shock (positive Fed funds innovation) should produce a hump-shaped negative response in the output gap, peaking around $h=4$–$8$ quarters. If LP and VAR(4) largely agree on the shape but the LP bands are wider, this is the efficiency-robustness trade-off working as expected. If VAR(1) diverges sharply from both, this confirms that lag truncation matters.

**What to look for in panel (b) — inflation:** The inflation response is more contentious. Under VAR(4) with this Cholesky ordering, you may see an initial positive response before inflation falls — the **price puzzle** (Sims, 1992). The LP inflation response is often less puzzling because LP does not impose the strong cross-equation restrictions of the VAR. If both LP and VAR(4) show the price puzzle, it suggests the puzzle is data-driven (missing the commodity price information variable) rather than a VAR artefact.

In [ ]:
# ── Figure 2: Fed funds own-response (monetary policy persistence) ────────────
# The own-response of the Fed funds rate measures how long a monetary policy
# shock persists. A rate rise that reverts within 4 quarters is very different
# from one that persists for 12+ quarters.

var4_ff  = irf_var4.orth_irfs[:, IDX_FF, IDX_FF]    # VAR(4) own-response
var1_ff  = irf_var1.orth_irfs[:, IDX_FF, IDX_FF]    # VAR(1) own-response
se_var4_ff = irf_var4.stderr()[:, IDX_FF, IDX_FF]

fig, ax = plt.subplots(figsize=(10, 5))

# VAR(4)
ax.fill_between(horizons,
                var4_ff - 1.96*se_var4_ff,
                var4_ff + 1.96*se_var4_ff,
                color=C_VAR4, alpha=ALPHA)
ax.plot(horizons, var4_ff, color=C_VAR4, linewidth=2.5,
        label=f'VAR({VAR_P_MAIN})')

# LP
ax.fill_between(horizons, lo_ff, hi_ff, color=C_LP, alpha=ALPHA)
ax.plot(horizons, lp_ff, color=C_LP, linewidth=2.5, linestyle='--',
        label=f'Local Projections  (L={LP_LAGS})')

# VAR(1)
ax.plot(horizons, var1_ff, color=C_VAR1, linewidth=2.0, linestyle=':',
        label=f'VAR({VAR_P_MISSP}) — misspecified')

ax.axhline(0, color='black', linewidth=1.0)
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle=':', alpha=0.7,
           label='Half-life reference (0.5)')
ax.set_xlabel('Quarters after shock')
ax.set_ylabel('Percentage points')
ax.set_xlim(0, IRF_HORIZON)
ax.set_xticks([0, 4, 8, 12, 16, 20])
ax.set_title(
    f'Figure 2: Own-Response of Federal Funds Rate\n'
    f'Persistence of a 1 pp monetary policy shock  |  {first_q}–{last_q}',
    fontsize=11
)
ax.legend()
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig2_ff_persistence.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** The own-response at $h=0$ is 1 by construction (a 1 pp shock raises the Fed funds rate by 1 pp on impact). The decline thereafter reflects mean-reversion. The **half-life** of the shock is the horizon at which the response crosses 0.5 — typically 4–6 quarters for the pre-crisis US data. If VAR(1) shows faster mean-reversion than LP, this confirms that the single-lag model understates monetary policy persistence.

---
## Step 5 — Numerical Summary at Selected Horizons

In [ ]:
# ── Summary table: IRF values at key horizons ─────────────────────────────────
# Reports the point estimate and 95% confidence interval for each method
# at h = 0, 4, 8, 12, 20 (impact, 1yr, 2yr, 3yr, 5yr)

KEY_HORIZONS = [0, 4, 8, 12, 20]

print('=' * 90)
print(f'LP vs VAR Comparison — {first_q} – {last_q}')
print('=' * 90)

for response_label, lp_irf, lp_lo, lp_hi, var4_irf, var4_lo, var4_hi, var1_irf in [
    ('Output Gap',
     lp_y,  lo_y,  hi_y,
     var4_y, var4_lo_y, var4_hi_y, var1_y),
    ('Inflation',
     lp_pi, lo_pi, hi_pi,
     var4_pi, var4_lo_pi, var4_hi_pi, var1_pi),
]:
    print(f'\nResponse: {response_label}  (shock = +1 pp Fed Funds Rate)')
    print(f'{"Horizon (h)":>12}  {"LP point":>10}  {"LP 95% CI":>22}  '
          f'{"VAR(4) pt":>10}  {"VAR(4) 95% CI":>22}  {"VAR(1)"}:>8')
    print('-' * 90)
    for h in KEY_HORIZONS:
        print(f'{h:>10}Q  '
              f'{lp_irf[h]:>+10.4f}  '
              f'[{lp_lo[h]:+.4f}, {lp_hi[h]:+.4f}]  '
              f'{var4_irf[h]:>+10.4f}  '
              f'[{var4_lo[h]:+.4f}, {var4_hi[h]:+.4f}]  '
              f'{var1_irf[h]:>+8.4f}')

print('\n' + '=' * 90)
print('Notes:')
print('  - All values in % response to a 1 pp increase in the Fed funds rate')
print('  - LP standard errors: Newey-West HAC, truncation = max(h+1, L)')
print(f'  - VAR({VAR_P_MAIN}) standard errors: delta-method (analytical approximation)')
print(f'  - VAR({VAR_P_MISSP}): no confidence bands shown (misspecified benchmark only)')

---
## Step 6 — Understanding LP Band Width vs VAR Band Width

In [ ]:
# ── Figure 3: Band width comparison ──────────────────────────────────────────
# One of the most discussed features of LP is that confidence bands widen
# quickly with horizon, while VAR bands grow more slowly (due to the parametric
# structure imposing cross-equation restrictions).
# This figure makes the width comparison explicit.

lp_width_y  = hi_y  - lo_y     # width of LP 95% CI for output gap
var_width_y = var4_hi_y - var4_lo_y   # width of VAR(4) 95% CI
lp_width_pi = hi_pi - lo_pi
var_width_pi= var4_hi_pi - var4_lo_pi

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, lp_w, var_w, title in [
    (axes[0], lp_width_y,  var_width_y,  '(a) Output Gap'),
    (axes[1], lp_width_pi, var_width_pi, '(b) Inflation'),
]:
    ax.plot(horizons, lp_w,  color=C_LP,   linewidth=2.2, linestyle='--',
            label='LP 95% CI width')
    ax.plot(horizons, var_w, color=C_VAR4, linewidth=2.2,
            label=f'VAR({VAR_P_MAIN}) 95% CI width')
    ax.set_xlabel('Horizon (quarters)')
    ax.set_ylabel('Width of 95% confidence interval')
    ax.set_title(title)
    ax.set_xlim(0, IRF_HORIZON)
    ax.set_xticks([0, 4, 8, 12, 16, 20])
    ax.legend()

fig.suptitle(
    'Figure 3: Efficiency Comparison — Width of 95% Confidence Intervals\n'
    'LP bands widen faster with horizon; VAR bands are narrower due to parametric restrictions',
    fontsize=12, fontweight='bold'
)
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig3_band_width.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** LP confidence bands typically widen faster than VAR bands as the horizon increases. This is the **efficiency cost of robustness**: by not imposing the VAR's cross-equation restrictions, LP loses efficiency. The ratio of LP-to-VAR band width growing with $h$ is the empirical signature of this trade-off. At short horizons ($h \leq 4$), the bands are often similar in width; at $h = 16$–$20$, LP bands can be 2–3× wider.

---
## Summary

| | LP | VAR(4) | VAR(1) |
|---|---|---|---|
| **Estimator** | Direct projection at each $h$ | MA inversion of companion matrix | Same, truncated |
| **Standard errors** | Newey-West HAC (accounts for MA($h$) errors) | Delta method | Delta method |
| **Band width** | Wider, especially at long $h$ | Narrower (parametric restrictions) | Narrower (over-restricted) |
| **Misspecification** | Robust — each regression uses only moment conditions at $h$ | Propagates through all horizons | Severe at $h > 4$ |
| **When to prefer** | Suspected VAR misspecification; nonlinear extensions | Correct model known; efficiency matters | Never |

## Exercises

**Exercise 1 (Lag robustness):** Change `LP_LAGS` from 4 to 2 and then to 8. How much do the LP point estimates move? Do the confidence bands change? This checks sensitivity to the control lag length.

**Exercise 2 (Cholesky ordering):** Swap the ordering to `['output_gap', 'fed_funds', 'inflation']` (Fed funds second, inflation last). Re-run the VAR IRFs. Does the price puzzle appear or disappear? How do the LP results change? LP is invariant to the Cholesky ordering — why?

**Exercise 3 (Extended sample):** Change `SAMPLE_END` to `'2019-12-31'` to include the ZLB era (2009–2015) but exclude COVID. Do the LP responses change materially? Does including near-zero interest rate periods affect the persistence of the shock (Figure 2)?

**Exercise 4 (Year-on-year vs quarter-on-quarter inflation):** Replace `df['inflation'] = df['log_deflator'].diff(4)` with `df['inflation'] = 4 * df['log_deflator'].diff(1)` (annualised quarterly inflation). Re-run LP. The confidence bands should narrow — why? (Hint: think about the MA structure of the regression error.)

**Exercise 5 (Romer-Romer shocks):** If you have access to the Romer & Romer (2004) narrative monetary policy shock series, substitute it for the Fed funds rate as the shock variable in `estimate_lp`. The narrative shock directly measures *exogenous* Fed actions, avoiding the endogeneity problem. Compare the IRFs to the Cholesky-identified results.

**Exercise 6 (State-dependent LP):** Split the sample into recession quarters (`output_gap < 0`) and expansion quarters (`output_gap >= 0`). Estimate LP separately for each state. Are monetary policy shocks more contractionary during recessions? (This is the nonlinear extension of LP developed by Jordà, Schularick & Taylor, 2015.)

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 3: Local Projections | Last updated: March 2026  
Data: FRED GDPC1, GDPDEF, FEDFUNDS (downloaded January 2026)  
Reference: Jordà, Ò. (2005). *Estimation and Inference of Impulse Responses by Local Projections.* AER, 95(1), 161–182.